<a href="https://colab.research.google.com/github/Rachitsingh13/Student-Prediction/blob/w4/week6yuva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import numpy as np
import pandas as pd

from flask import Flask, request, jsonify

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


FILE_PATH = "/content/collegePlace.csv"
MODEL_PATH = "student_placement_model.pkl"


numeric_features = [
    "Age",
    "Internships",
    "CGPA",
    "Hostel",
    "HistoryOfBacklogs"
]

categorical_features = [
    "Gender",
    "Stream"
]


def load_data():

    df = pd.read_csv(FILE_PATH)

    df = df.drop_duplicates()

    df["PlacedOrNot"] = df["PlacedOrNot"].astype(int)

    return df


def create_pipeline():

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ])

    classifier = RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42
    )

    pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            classifier
        )
    ])

    return pipeline


def train_model():

    df = load_data()

    X = df.drop(
        columns=["PlacedOrNot"]
    )

    y = df["PlacedOrNot"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    model = create_pipeline()

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    metrics = {
        "accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        ),
        "average_precision": average_precision_score(
            y_test,
            probabilities
        )
    }

    print("\nFinal Model Performance")

    for key, value in metrics.items():
        print(
            key,
            ":",
            round(value, 4)
        )

    print("\nConfusion Matrix")

    print(
        confusion_matrix(
            y_test,
            predictions
        )
    )

    print("\nClassification Report")

    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

    joblib.dump(
        model,
        MODEL_PATH
    )

    print(
        "\nModel saved as:",
        MODEL_PATH
    )

    return model, metrics


def load_model():

    if not os.path.exists(
        MODEL_PATH
    ):
        model, _ = train_model()
        return model

    return joblib.load(
        MODEL_PATH
    )


def validate_input(data):

    required_fields = [
        "Age",
        "Gender",
        "Stream",
        "Internships",
        "CGPA",
        "Hostel",
        "HistoryOfBacklogs"
    ]

    for field in required_fields:

        if field not in data:

            return False, (
                "Missing field: "
                + field
            )

    try:

        age = float(
            data["Age"]
        )

        internships = float(
            data["Internships"]
        )

        cgpa = float(
            data["CGPA"]
        )

        hostel = int(
            data["Hostel"]
        )

        backlogs = int(
            data["HistoryOfBacklogs"]
        )

        if age <= 0:
            return False, "Invalid Age"

        if internships < 0:
            return False, "Invalid Internships"

        if cgpa < 0 or cgpa > 10:
            return False, "CGPA must be between 0 and 10"

        if hostel not in [0, 1]:
            return False, "Hostel must be 0 or 1"

        if backlogs not in [0, 1]:
            return False, "HistoryOfBacklogs must be 0 or 1"

    except:

        return False, "Invalid input data type"

    return True, ""


def prepare_input(data):

    return pd.DataFrame([
        {
            "Age": float(
                data["Age"]
            ),

            "Gender": data[
                "Gender"
            ],

            "Stream": data[
                "Stream"
            ],

            "Internships": float(
                data["Internships"]
            ),

            "CGPA": float(
                data["CGPA"]
            ),

            "Hostel": int(
                data["Hostel"]
            ),

            "HistoryOfBacklogs": int(
                data["HistoryOfBacklogs"]
            )
        }
    ])


def predict_student(
    model,
    data
):

    valid, message = validate_input(
        data
    )

    if not valid:

        return {
            "success": False,
            "error": message
        }

    input_data = prepare_input(
        data
    )

    prediction = model.predict(
        input_data
    )[0]

    probability = model.predict_proba(
        input_data
    )[0][1]

    result = (
        "Placed"
        if prediction == 1
        else "Not Placed"
    )

    return {
        "success": True,
        "prediction": int(
            prediction
        ),
        "result": result,
        "placement_probability": round(
            float(probability),
            4
        )
    }


def run_integration_test(
    model
):

    test_student = {
        "Age": 22,
        "Gender": "Male",
        "Stream": "Computer Science",
        "Internships": 1,
        "CGPA": 8,
        "Hostel": 1,
        "HistoryOfBacklogs": 0
    }

    result = predict_student(
        model,
        test_student
    )

    print("\nIntegration Test")

    print(
        result
    )

    if result["success"]:

        print(
            "Integration Test: PASSED"
        )

    else:

        print(
            "Integration Test: FAILED"
        )


def run_batch_prediction(
    model,
    file_path
):

    data = pd.read_csv(
        file_path
    )

    predictions = model.predict(
        data
    )

    probabilities = model.predict_proba(
        data
    )[:, 1]

    data["PredictedPlacement"] = predictions

    data["PlacementProbability"] = probabilities

    data["Result"] = np.where(
        predictions == 1,
        "Placed",
        "Not Placed"
    )

    data.to_csv(
        "placement_predictions.csv",
        index=False
    )

    print(
        "\nBatch prediction completed."
    )

    return data


model = load_model()

run_integration_test(
    model
)


app = Flask(
    __name__
)


@app.route(
    "/",
    methods=["GET"]
)
def home():

    return jsonify({
        "message":
        "Student Placement Prediction API",
        "status":
        "running"
    })


@app.route(
    "/health",
    methods=["GET"]
)
def health():

    return jsonify({
        "status":
        "healthy",
        "model":
        "Tuned Random Forest"
    })


@app.route(
    "/predict",
    methods=["POST"]
)
def predict():

    try:

        data = request.get_json()

        result = predict_student(
            model,
            data
        )

        if not result["success"]:

            return jsonify(
                result
            ), 400

        return jsonify(
            result
        )

    except Exception as e:

        return jsonify({
            "success": False,
            "error":
            str(e)
        }), 500


@app.route(
    "/predict-batch",
    methods=["POST"]
)
def predict_batch():

    try:

        data = request.get_json()

        if not isinstance(
            data,
            list
        ):

            return jsonify({
                "success": False,
                "error":
                "Input must be a list of students"
            }), 400

        results = []

        for student in data:

            result = predict_student(
                model,
                student
            )

            results.append(
                result
            )

        return jsonify({
            "success": True,
            "predictions":
            results
        })

    except Exception as e:

        return jsonify({
            "success": False,
            "error":
            str(e)
        }), 500


if __name__ == "__main__":

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False
    )


Final Model Performance
accuracy : 0.7456
precision : 0.8246
recall : 0.7121
f1_score : 0.7642
roc_auc : 0.8228
average_precision : 0.8999

Confusion Matrix
[[76 20]
 [38 94]]

Classification Report
              precision    recall  f1-score   support

           0       0.67      0.79      0.72        96
           1       0.82      0.71      0.76       132

    accuracy                           0.75       228
   macro avg       0.75      0.75      0.74       228
weighted avg       0.76      0.75      0.75       228


Model saved as: student_placement_model.pkl

Integration Test
{'success': True, 'prediction': 1, 'result': 'Placed', 'placement_probability': 0.9414}
Integration Test: PASSED
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
